# PDHG FFHQ-100 Single Run In Colab

This notebook runs one configurable PDHG experiment over a fixed 100-image FFHQ slice.

Use it when you want to evaluate one inverse problem with one chosen parameter set, rather than launch a sweep.

## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

Prefer `A100` or `H100`. If `batch_size=100` is too large for the runtime, reduce it to `50` or `25` and rerun.

In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_single_run_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
RUN_NAME = "PDHG_Single_Run"  #@param {type:"string"}
CONFIG_NAME = "default_ffhq.yaml"  #@param {type:"string"}

INVERSE_TASK = "phase_retrieval"  #@param ["phase_retrieval", "phase_retrieval_explicit", "inpainting", "inpainting_explicit", "motion_blur", "motion_blur_explicit", "gaussian_blur", "gaussian_blur_explicit", "down_sampling", "down_sampling_explicit", "hdr", "hdr_explicit"]

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}

NUM_STEPS = 500  #@param {type:"integer"}
MAX_ITER = 500  #@param {type:"integer"}
SIGMA_MAX = 27.0  #@param {type:"number"}
SIGMA_MIN = 0.075  #@param {type:"number"}
TAU = 0.01  #@param {type:"number"}
SIGMA_DUAL = 1600.0  #@param {type:"number"}
MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}
DENOISE_FINAL_STEP = "tweedie"  #@param ["tweedie", "ode"]

DYS_GAMMA = 0.0075  #@param {type:"number"}
LAMBDA_START = 1.0  #@param {type:"number"}
LAMBDA_END = 1.0  #@param {type:"number"}
LAMBDA_WARMUP = 0  #@param {type:"integer"}

EVAL_METRICS = "psnr;ssim;lpips"  #@param {type:"string"}
SAVE_SAMPLES = False  #@param {type:"boolean"}
SAVE_TRAJ = False  #@param {type:"boolean"}
SAVE_TRAJ_RAW_DATA = False  #@param {type:"boolean"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

# Optional extra Hydra overrides, separated by semicolons.
# Example for inpainting: inverse_task.operator.mask_type=box;inverse_task.operator.mask_len_range=[128,129]
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            repo_dir.as_posix(),
        ],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repo ready: {repo_dir}")


In [ ]:
#@title Install Colab Dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
ckpt_path = Path("pretrained-models/ffhq_10m.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

if ckpt_path.exists():
    print(f"Checkpoint already present: {ckpt_path}")
else:
    subprocess.run(
        [
            "gdown",
            "--id",
            "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh",
            "-O",
            ckpt_path.as_posix(),
        ],
        check=True,
    )
    print(f"Downloaded checkpoint to: {ckpt_path}")


In [ ]:
#@title Build Single-Run Command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

def parse_list(text: str):
    items = []
    for raw_item in text.replace("\n", ";").split(";"):
        raw_item = raw_item.strip()
        if raw_item:
            items.append(raw_item)
    return items

metric_list = parse_list(EVAL_METRICS)
if not metric_list:
    raise ValueError("EVAL_METRICS must contain at least one metric name.")

extra_overrides = parse_list(EXTRA_HYDRA_OVERRIDES)
session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
run_tag = f"{INVERSE_TASK}_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"
save_root = repo_dir / "results" / "single_runs" / run_tag
hydra_root = repo_dir / "outputs" / "single_runs" / run_tag
run_aux_root = repo_dir / "single_runs"
latest_log_path = run_aux_root / f"{run_tag}.log"
latest_pid_path = run_aux_root / f"{run_tag}.pid"
data_end_idx = DATA_START_IDX + TOTAL_IMAGES
eval_fn_override = f"eval_fn_list=[{','.join(metric_list)}]"

run_cmd = [
    PYTHON_BIN,
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    "sampler=edm_pdhg",
    f"inverse_task={INVERSE_TASK}",
    f"name={run_name}",
    f"seed={SEED}",
    "gpu=0",
    "wandb=false",
    "show_config=false",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    f"save_traj={'true' if SAVE_TRAJ else 'false'}",
    f"save_traj_raw_data={'true' if SAVE_TRAJ_RAW_DATA else 'false'}",
    f"total_images={TOTAL_IMAGES}",
    f"batch_size={BATCH_SIZE}",
    "num_runs=1",
    f"inverse_task.operator.sigma={MEASUREMENT_SIGMA}",
    f"sampler.annealing_scheduler_config.num_steps={NUM_STEPS}",
    f"sampler.annealing_scheduler_config.sigma_max={SIGMA_MAX}",
    f"sampler.annealing_scheduler_config.sigma_min={SIGMA_MIN}",
    f"inverse_task.admm_config.max_iter={MAX_ITER}",
    f"inverse_task.admm_config.pdhg.tau={TAU}",
    f"inverse_task.admm_config.pdhg.sigma_dual={SIGMA_DUAL}",
    f"inverse_task.admm_config.denoise.final_step={DENOISE_FINAL_STEP}",
    f"inverse_task.admm_config.dys.gamma={DYS_GAMMA}",
    "inverse_task.admm_config.dys.lambda_schedule.activate=true",
    f"inverse_task.admm_config.dys.lambda_schedule.start={LAMBDA_START}",
    f"inverse_task.admm_config.dys.lambda_schedule.end={LAMBDA_END}",
    f"inverse_task.admm_config.dys.lambda_schedule.warmup={LAMBDA_WARMUP}",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
    eval_fn_override,
    f"data.image_root_path={drive_data_dir.as_posix()}",
    f"data.start_idx={DATA_START_IDX}",
    f"data.end_idx={data_end_idx}",
]

run_cmd.extend(extra_overrides)
run_cmd.extend([
    f"save_dir={save_root.as_posix()}",
    f"hydra.run.dir={hydra_root.as_posix()}",
])

print(f"Run tag: {run_tag}")
print(f"Save root: {save_root}")
print(f"Dataset slice: [{DATA_START_IDX}, {data_end_idx})")
print(f"Metrics: {metric_list}")
if extra_overrides:
    print(f"Extra overrides: {extra_overrides}")

print("\nCommand:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

last_context = {
    "run_tag": run_tag,
    "run_name": run_name,
    "save_root": save_root.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_cmd": run_cmd,
}

run_aux_root.mkdir(parents=True, exist_ok=True)
context_path = run_aux_root / f"{run_tag}.context.json"
with context_path.open("w", encoding="utf-8") as handle:
    json.dump(last_context, handle, indent=2)
print(f"Context saved to: {context_path}")


In [ ]:
#@title Launch The Run In Background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
latest_log_path = Path(last_context["latest_log_path"])
latest_pid_path = Path(last_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

latest_pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {latest_log_path}")
print(f"Save root: {last_context['save_root']}")


In [ ]:
#@title Show Recent Log Lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
tail = lines[-int(LOG_TAIL_LINES):]
print("\n".join(tail) if tail else "<log is empty>")


In [ ]:
#@title Show Results And Artifacts
import json
from pathlib import Path

save_root = Path(last_context["save_root"])
metrics_matches = sorted(save_root.rglob("metrics.json"))
eval_matches = sorted(save_root.rglob("eval.md"))
grid_matches = sorted(save_root.rglob("grid_results.png"))

print("Artifacts:\n")
print(f"save_root: {save_root}")
print(f"metrics.json matches: {[p.as_posix() for p in metrics_matches]}")
print(f"eval.md matches: {[p.as_posix() for p in eval_matches]}")
print(f"grid_results.png matches: {[p.as_posix() for p in grid_matches]}")

if eval_matches:
    eval_path = eval_matches[0]
    print("\neval.md:\n")
    print(eval_path.read_text(encoding="utf-8", errors="ignore"))
else:
    print("\nNo eval.md found yet.")

if metrics_matches:
    metrics_path = metrics_matches[0]
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\nmetrics.json:\n")
    print(json.dumps(metrics, indent=2))
else:
    print("\nNo metrics.json found yet.")


In [ ]:
#@title Copy Run Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["save_root"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
]

for src in targets:
    if not src.exists():
        print(f"Skipping missing path: {src}")
        continue

    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
